# Styler Object and Customising the Display
# Formatting the Display
## Formatting Values

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
    {
        "strings": ["Adam", "Mike"],
        "ints": [1, 3],
        "floats": [1.123, 1000.23],
    }
)

(
    df.style.format(precision=3, thousands=".", decimal=",")
    .format_index(str.upper, axis=1)
    .relabel_index(["row 1", "row 2"], axis=0)
)

,STRINGS,INTS,FLOATS
row 1,Adam,1,"1,123"
row 2,Mike,3,"1.000,230"


Using the formatting functions while still relying on the underlying data for indexing and calculations.

In [2]:
weather_df = pd.DataFrame(
    np.random.default_rng().random((10, 2)) * 5,
    index=pd.date_range(start="2021-01-01", periods=10),
    columns=["Tokyo", "Beijing"],
)


def rain_condition(v):
    if v < 1.75:
        return "Dry"
    elif v < 2.75:
        return "Rain"
    return "Heavy Rain"


def make_pretty(styler: pd.io.formats.style.Styler):
    styler.set_caption("Weather Conditions")
    styler.format(rain_condition)
    styler.format_index(lambda v: v.strftime("%A"))  # extracts the full name of the day of the week
    styler.background_gradient(axis=None, vmin=1, vmax=5, cmap="YlGnBu")

    return styler

In [3]:
weather_df

,Tokyo,Beijing
2021-01-01,4.367875,2.131675
2021-01-02,1.831372,4.223505
2021-01-03,3.141755,3.997238
2021-01-04,3.384421,3.156182
2021-01-05,3.354022,4.627157
2021-01-06,2.243723,4.450306
2021-01-07,3.188155,3.572518
2021-01-08,4.583499,4.887819
2021-01-09,1.819668,2.502425
2021-01-10,2.672868,0.698376


In [4]:
weather_df.loc["2021-01-04":"2021-01-08"].style.pipe(make_pretty)

,Tokyo,Beijing
Monday,Heavy Rain,Heavy Rain
Tuesday,Heavy Rain,Heavy Rain
Wednesday,Rain,Heavy Rain
Thursday,Heavy Rain,Heavy Rain
Friday,Heavy Rain,Heavy Rain


## Hiding Data

In [5]:
df = pd.DataFrame(np.random.default_rng().standard_normal((5, 5)))
(df.style.hide(subset=[0, 2, 4], axis=0).hide(subset=[0, 2, 4], axis=1))

,1,3
1,0.797743,-1.390001
3,-0.085039,0.839300


In [6]:
show = [0, 2, 4]
(
    df.style.hide(
        [row for row in df.index if row not in show],
        axis=0,
    ).hide(
        [col for col in df.columns if col not in show],
        axis=1,
    )
)

,0,2,4
0,0.127311,1.232239,-0.482278
2,-0.412743,-1.200773,0.803654
4,1.683901,-0.590373,0.165128


## Concatenating DataFrame Outputs

In [7]:
# Independently styling two DataFrames and concatenating them
# Note that the concatenated table preserves the independent styles:
summary_styler = (
    df.agg(
        ["sum", "mean"],
    )
    .style.format(precision=3)
    .relabel_index(["Sum", "Average"])
)
df.style.format(precision=1).concat(summary_styler)

,0,1,2,3,4
0,0.1,1.0,1.2,0.9,-0.5
1,0.4,0.8,-1.7,-1.4,-0.1
2,-0.4,-0.0,-1.2,-0.8,0.8
3,0.3,-0.1,-2.0,0.8,0.2
4,1.7,-1.2,-0.6,0.6,0.2
Sum,2.017,0.498,-4.180,0.167,0.528
Average,0.403,0.100,-0.836,0.033,0.106


# Styler Object and HTML

In [8]:
idx = pd.Index(["Tumour (Positive)", "Non-Tumour (Negative)"], name="Actual Label:")
cols = pd.MultiIndex.from_product(
    [
        ["Decision Tree", "Regression", "Random"],
        ["Tumour", "Non-Tumour"],
    ],
    names=["Model:", "Predicted"],
)
df = pd.DataFrame(
    [
        [38.0, 2.0, 18.0, 22.0, 21, np.nan],
        [19, 439, 6, 452, 226, 232],
    ],
    index=idx,
    columns=cols,
)
df.style

In [9]:
s = df.style.format("{:.0f}").hide(
    [("Random", "Tumour"), ("Random", "Non-Tumour")],
    axis="columns",
)
s

# Methods to Add Styles

# Table Styles

In [10]:
# For row hover use <tr> instead of <td>
cell_hover = {
    "selector": "td:hover",
    "props": [("background-color", "#FFFFB3"), ("color", "black")],
}
index_names = {
    "selector": ".index_name",
    "props": "font-style: italic; color: darkgrey; font-weight: normal;",
}
headers = {
    "selector": "th:not(.index_name)",
    "props": "background-color: #000066; color: white;",
}

s.set_table_styles([cell_hover, index_names, headers])

In [11]:
# Preserve the previous styles we have set, by specifying overwrite=False
s.set_table_styles(
    [
        {"selector": "th.col_heading", "props": "text-align: center;"},
        {"selector": "th.col_heading.level0", "props": "font-size: 1.5em;"},
        {"selector": "td", "props": "text-align: center; font-weight: bold"},
    ],
    overwrite=False,
)

In [12]:
# We can also pass in a dict with row or column keys
s.set_table_styles(
    {
        ("Regression", "Tumour"): [
            {"selector": "th", "props": "border-left: 1px solid white"},
            {"selector": "td", "props": "border-left:1px solid #000066"},
        ]
    },
    overwrite=False,
    axis=0,
)

# Setting Classes and Linking to External CSS

## Table Attributes

In [13]:
# Add a class to the main <table>
out = s.set_table_attributes("class='my-table-cls'").to_html()
print(out[out.find("<table") :][:109])

<table id="T_29c97" class='my-table-cls'>
  <thead>
    <tr>
      <th class="index_name level0" >Model:</th>


## Data Cell CSS Classes

In [14]:
# Create our own CSS classes internally and add them to table style
s.set_table_styles(
    [
        # create internal CSS classes
        {"selector": ".true", "props": "background-color: #E6FFE6;"},
        {"selector": ".false", "props": "background-color: #FFE6E6;"},
    ],
    overwrite=False,
)
cell_color = pd.DataFrame(
    [
        ["true ", "false ", "true ", "false "],
        ["false ", "true ", "false ", "true "],
    ],
    index=df.index,
    columns=df.columns[:4],
)

s.set_td_classes(cell_color)

# Styler Functions
## Acting on Data

In [15]:
df2 = pd.DataFrame(
    np.random.default_rng(0).standard_normal((10, 4)),
    columns=list("ABCD"),
)
df2.style

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [16]:
def style_negative(v, props=""):
    """Colors text red, if it's negative"""
    return props if v < 0 else None


s2 = df2.style.map(
    style_negative,
    props="color: red;",
).map(
    lambda v: "opacity: 20%;" if (v < 0.3) and (v > -0.3) else None,
)
s2

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [17]:
def highlight_max(s, props=""):
    """Highlight the maximum value in a column"""
    return np.where(s == np.nanmax(s.values), props, "")


s2.apply(highlight_max, props="color:white; background-color: darkblue;", axis=0)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [18]:
# Highlight the DataFrame maximum in purple
# Highlight row maximums in pink
s2.apply(highlight_max, props="color: white;background-color: pink;", axis=1).apply(
    highlight_max, props="color: white;background-color: purple", axis=None
)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


## Acting on the Index and Column Headers

In [19]:
s2.map_index(lambda v: "color: pink;" if v > 4 else "color: darkblue;", axis=0)
s2.apply_index(lambda s: np.where(s.isin(["A", "B"]), "color: pink;", "color: darkblue;"), axis=1)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


# Tooltips and Captions

In [20]:
s.set_caption(
    "Confusion matrix for multiple cancer prediction models.",
).set_table_styles(
    [
        {
            "selector": "caption",
            "props": "caption-side: bottom; font-size: 1.25em;",
        }
    ],
    overwrite=False,
)

In [21]:
tt = pd.DataFrame(
    [
        [
            "This model has a very strong true positive rate (TP)",
            "This model's total number of false negatives is too high (FN)",
        ]
    ],
    index=["Tumour (Positive)"],
    columns=df.columns[[0, 3]],
)
s.set_tooltips(
    tt,
    props="visibility: hidden; position: absolute; z-index: 1; "
    "border: 1px solid #000066;"
    "background-color: white; color: #000066; font-size: 0.8em;"
    "transform: translate(0px,-24px); padding: 0.6em;"
    "border-radius: 0.5em;",
)

In [22]:
s.set_table_styles(
    [  # create internal CSS classes
        {"selector": ".border-red", "props": "border: 2px dashed red;"},
        {"selector": ".border-green", "props": "border: 2px dashed green;"},
    ],
    overwrite=False,
)
cell_border = pd.DataFrame(
    [
        ["border-green ", " ", " ", "border-red "],
        [" ", " ", " ", " "],
    ],
    index=df.index,
    columns=df.columns[:4],
)
s.set_td_classes(cell_color + cell_border)

# Finer Control with Slicing

In [23]:
df3 = pd.DataFrame(
    np.random.default_rng().standard_normal((4, 4)),
    pd.MultiIndex.from_product([["A", "B"], ["r1", "r2"]]),
    columns=["c1", "c2", "c3", "c4"],
)
df3

c1        c2        c3        c4
A r1 -0.071258  0.099254 -0.756548 -1.952248
  r2  0.246480  1.461404 -0.573124  1.054066
B r1 -1.208920 -1.228511 -1.532927 -1.442870
  r2 -1.278784 -0.378404  0.610278  1.839265

In [24]:
# Highlight maximums in third and fourth columns
# Highlight sliced region in yellow
slice_ = ["c3", "c4"]
df3.style.apply(highlight_max, props="color: red;", axis=0, subset=slice_).set_properties(
    **{"background-color": "#FFFFB3"}, subset=slice_
)

In [25]:
# Index across both dimensions
# Take all labels from level 0 and only label "r1" from level 1;
# Then take only columns c2, c3 and c4
idx = pd.IndexSlice
slice_ = idx[idx[:, "r1"], idx["c2":"c4"]]

df3.style.apply(
    highlight_max,
    props="color: red;",
    axis=0,
    subset=slice_,
).set_properties(
    **{"background-color": "#FFFFB3"},
    subset=slice_,
)

In [26]:
slice_ = idx[idx[:, "r2"], :]
df3.style.apply(
    highlight_max,
    props="color: red;",
    axis=1,
    subset=slice_,
).set_properties(**{"background-color": "#FFFFB3"}, subset=slice_)

In [27]:
# Highlight maximum across columns 2 and 4 BUT only if the
# sum of columns 1 and 2 is less than -2.0
slice_ = idx[idx[(df3["c1"] + df3["c3"]) < -2.0], ["c2", "c4"]]
df3.style.apply(
    highlight_max,
    props="color: red;",
    axis=1,
    subset=slice_,
).set_properties(
    **{"background-color": "#FFFFB3"},
    subset=slice_,
)

# Optimization

## 1. Remove UUID and cell_ids

In [28]:
from pandas.io.formats.style import Styler

df4 = pd.DataFrame([[1, 2], [3, 4]])
s4 = Styler(df4, uuid_len=0, cell_ids=False)

## 2. Use Table Styles

In [29]:
props = "font-family: 'Times New Roman', Times, serif; color: #E83E8C; font-size: 1.3em;"
df4.style.set_table_styles([{"selector": "td.col1", "props": props}])

,0,1
0,1,2
1,3,4


## 3. Set Classes Instead of Using Styler Functions

In [30]:
build = lambda x: pd.DataFrame(x, index=df2.index, columns=df2.columns)
cls1 = build(df2.apply(highlight_max, props="cls-1 ", axis=0))
cls2 = build(df2.apply(highlight_max, props="cls-2 ", axis=1, result_type="expand").values)
cls3 = build(highlight_max(df2, props="cls-3 "))
df2.style.set_table_styles(
    [
        {"selector": ".cls-1", "props": "color:white;background-color:darkblue;"},
        {"selector": ".cls-2", "props": "color:white;background-color:pink;"},
        {"selector": ".cls-3", "props": "color:white;background-color:purple;"},
    ]
).set_td_classes(cls1 + cls2 + cls3)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


## 4. Don't Use Tooltips
## 5. If every byte counts, use string replacement

In [31]:
my_css = {
    "row_heading": "",
    "col_heading": "",
    "index_name": "",
    "col": "c",
    "row": "r",
    "col_trim": "",
    "row_trim": "",
    "level": "l",
    "data": "",
    "blank": "",
}
html = Styler(df4, uuid_len=0, cell_ids=False)
html.set_table_styles(
    [
        {"selector": "td", "props": props},
        {"selector": ".c1", "props": "color:green;"},
        {"selector": ".l0", "props": "color:blue;"},
    ],
    css_class_names=my_css,
)
print(html.to_html())

<style type="text/css">
#T_ td {
  font-family: 'Times New Roman', Times, serif;
  color: #E83E8C;
  font-size: 1.3em;
}
#T_ .c1 {
  color: green;
}
#T_ .l0 {
  color: blue;
}
</style>
<table id="T_">
  <thead>
    <tr>
      <th class=" l0" >&nbsp;</th>
      <th class=" l0 c0" >0</th>
      <th class=" l0 c1" >1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th class=" l0 r0" >0</th>
      <td class=" r0 c0" >1</td>
      <td class=" r0 c1" >2</td>
    </tr>
    <tr>
      <th class=" l0 r1" >1</th>
      <td class=" r1 c0" >3</td>
      <td class=" r1 c1" >4</td>
    </tr>
  </tbody>
</table>



In [32]:
html

,0,1
0,1,2
1,3,4


# Builtin Styles

## Highlight Null

In [33]:
df2.iloc[0, 2] = np.nan
df2.iloc[4, 3] = np.nan
df2.loc[:4].style.highlight_null(color="#BA2C1F")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


## Highlight Min or Max

In [34]:
df2.loc[:4].style.highlight_max(
    axis=1,
    props=("color: white; font-weight: bold; background: darkblue;"),
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


## Highlight Between

In [35]:
left = pd.Series([1.0, 0.0, 1.0], index=["A", "B", "D"])
df2.loc[:4].style.highlight_between(
    left=left,
    right=1.5,
    axis=1,
    props="color: white; background:purple;",
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


## Highlight Quantile

In [36]:
df2.loc[:4].style.highlight_quantile(q_left=0.85, axis=None, color="yellow")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


## Background Gradient and Text Gradient

In [37]:
import seaborn as sns

cm = sns.light_palette("green", as_cmap=True)
df2.style.background_gradient(cmap=cm)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [38]:
df2.style.text_gradient(cmap=cm)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


## Set Properties

In [39]:
df2.loc[:4].style.set_properties(
    **{
        "background": "black",
        "color": "lawngreen",
        "border-color": "white",
    }
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


## Bar Charts

In [40]:
df2.style.bar(subset=["A", "B"], color="#D65F5F")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [41]:
df2.style.format("{:.3f}", na_rep="").bar(
    align=0,
    vmin=-2.5,
    vmax=2.5,
    cmap="bwr",
    height=50,
    width=60,
    props="width: 120px; border-right: 1px solid black;",
).text_gradient(
    cmap="bwr",
    vmin=-2.5,
    vmax=2.5,
)

,A,B,C,D
0,0.126,-0.132,,0.105
1,-0.536,0.362,1.304,0.947
2,-0.704,-1.265,-0.623,0.041
3,-2.325,-0.219,-1.246,-0.732
4,-0.544,-0.316,0.412,
5,-0.129,1.366,-0.665,0.352
6,0.903,0.094,-0.743,-0.922
7,-0.458,0.220,-1.010,-0.209
8,-0.159,0.541,0.215,0.355
9,-0.654,-0.130,0.784,1.493


# Sharing Styles

In [42]:
# The style we want to export
style1 = (
    df2.style.map(style_negative, props="color: red;")
    .map(lambda v: "opacity: 20%;" if (v < 0.3) and (v > -0.3) else None)
    .set_table_styles([{"selector": "th", "props": "color: blue;"}])
    .hide(axis="index")
)

style1

A,B,C,D
0.125730,-0.132105,nan,0.104900
-0.535669,0.361595,1.304000,0.947081
-0.703735,-1.265421,-0.623274,0.041326
-2.325031,-0.218792,-1.245911,-0.732267
-0.544259,-0.316300,0.411631,nan
-0.128535,1.366463,-0.665195,0.351510
0.903470,0.094012,-0.743499,-0.921725
-0.457726,0.220195,-1.009618,-0.209176
-0.159225,0.540846,0.214659,0.355373
-0.653829,-0.129614,0.783975,1.493431


In [43]:
style2 = df3.style
style2.use(style1.export())

style2

c1,c2,c3,c4
-0.071258,0.099254,-0.756548,-1.952248
0.246480,1.461404,-0.573124,1.054066
-1.208920,-1.228511,-1.532927,-1.442870
-1.278784,-0.378404,0.610278,1.839265


# Limitations

# Other Fun and Useful Stuff
## Widgets

In [46]:
from ipywidgets import widgets


@widgets.interact
def f(h_neg=(0, 359, 1), h_pos=(0, 359), s=(0.0, 99.9), l_post=(0.0, 99.9)):
    return df2.style.background_gradient(
        cmap=sns.palettes.diverging_palette(h_neg=h_neg, h_pos=h_pos, s=s, l=l_post, as_cmap=True)
    )

interactive(children=(IntSlider(value=179, description='h_neg', max=359), IntSlider(value=179, description='h_…

## Magnify

In [51]:
def magnify():
    return [
        {"selector": "th", "props": [("font-size", "4pt")]},
        {"selector": "td", "props": [("padding", "0em 0em")]},
        {"selector": "th:hover", "props": [("font-size", "12pt")]},
        {"selector": "tr:hover td:hover", "props": [("max-width", "200px"), ("font-size", "12pt")]},
    ]

In [52]:
cmap = sns.diverging_palette(5, 250, as_cmap=True)
bigdf = pd.DataFrame(np.random.default_rng(25).standard_normal((20, 25))).cumsum()
bigdf.style.background_gradient(cmap, axis=1).set_properties(**{"max-width": "80px", "font-size": "1pt"}).set_caption(
    "Hover to magnify"
).format(precision=2).set_table_styles(magnify())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,0.35,-0.00,-0.53,-2.28,0.02,0.93,1.04,-0.54,2.23,1.94,-0.23,-0.16,0.96,-0.25,0.22,1.22,1.19,-0.53,-0.46,-0.55,-0.01,0.07,-0.39,0.77,0.05
1,0.37,-1.97,-1.92,-2.97,0.38,0.06,2.60,-0.39,2.49,2.06,0.26,0.77,0.41,1.31,-0.55,1.74,0.98,-0.70,0.21,-0.35,1.59,-0.43,0.13,1.36,2.16
2,1.98,-1.27,-0.24,-2.12,-0.40,0.39,3.83,-0.91,4.01,2.42,-0.36,1.23,-1.16,0.62,-1.60,2.24,-0.04,-2.26,0.52,-0.83,0.87,-1.03,1.17,1.53,1.41
3,4.78,-0.78,1.67,-1.07,0.96,-0.48,3.26,-0.27,3.93,2.50,0.72,-1.07,-1.49,2.16,-2.07,2.28,0.34,-1.94,0.83,-1.76,1.20,-1.63,2.75,0.90,1.67
4,4.58,0.35,1.79,-0.36,0.40,-0.86,2.31,-0.44,3.65,1.36,-0.53,-0.91,-1.67,1.45,-2.99,1.17,0.44,-2.48,0.20,-2.32,0.73,-1.94,2.84,-0.86,1.41
5,3.38,-0.02,3.04,-2.28,-0.89,-1.38,0.01,1.95,3.01,2.98,-0.62,0.27,-0.86,1.15,-2.49,0.51,-0.04,-0.99,-0.67,-3.24,3.28,-2.33,3.14,0.78,0.92
6,2.81,1.80,3.02,-2.01,-1.23,-0.85,-1.57,2.12,4.01,3.94,-1.13,0.91,1.28,1.27,-1.81,0.15,-1.77,-0.25,-1.63,-3.94,3.34,-2.35,3.01,1.90,1.05
7,2.85,0.72,3.92,-3.29,-3.49,-1.67,-0.87,2.46,5.28,4.34,-0.76,1.24,0.64,0.81,-2.27,0.34,-2.71,-0.67,-1.35,-4.01,3.06,-2.55,1.78,0.54,0.25
8,3.80,1.51,4.19,-2.67,-2.71,-4.04,-1.42,3.30,7.04,4.52,0.96,2.40,0.78,-0.24,-3.28,0.28,-2.51,-0.74,-0.74,-3.44,3.11,-2.12,2.07,-1.35,-1.03
9,4.11,2.27,4.29,-2.97,-2.50,-3.58,-1.25,1.32,7.36,5.25,1.08,4.60,-0.64,-2.05,0.09,-0.72,-4.96,-1.31,0.09,-4.16,2.68,-1.37,4.02,-0.59,-0.72


## Sticky Headers

In [53]:
bigdf = pd.DataFrame(np.random.default_rng().standard_normal((16, 100)))
bigdf.style.set_sticky(axis="index")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
0,-0.130105,-0.590540,-0.662313,-0.323766,1.974304,0.900250,0.145534,-0.200769,0.626681,0.295408,1.227525,-0.200313,-2.205787,-0.992848,0.825751,-0.502273,-0.654839,0.721447,1.239156,1.224467,-1.730704,0.686905,-0.013704,0.969517,0.536365,-1.188404,0.659401,-1.032033,0.463313,0.745238,-0.579297,-0.692674,-0.357433,-0.312813,-0.846201,0.463105,0.159121,-0.832525,-1.261996,-0.316836,-2.657442,1.846220,-0.934894,-0.228691,-0.054666,0.514116,-1.709327,-0.876911,0.952106,-2.244479,0.023237,0.292280,0.407374,-1.318612,-0.053729,1.470978,0.638143,0.949179,-0.704968,-2.239832,-1.933060,0.280568,-0.328429,1.039968,-0.569093,0.576166,-0.659375,-0.110877,0.994780,0.547178,-0.079535,-1.754416,0.330318,-2.619021,-1.104861,-1.052821,-0.728535,-0.191295,1.059860,1.522250,-0.935956,0.292301,0.774827,-1.834184,0.178916,0.303580,-1.177009,0.659014,-0.954375,-0.822718,-0.781007,0.862532,-0.574996,0.561122,0.381782,-1.090478,1.227368,0.548228,0.331776,0.411503
1,-1.002286,1.541776,-0.129867,0.403624,0.356063,-0.481453,0.627777,-2.248870,-1.860279,-0.963445,0.350644,-1.851134,0.620576,0.299272,-0.997679,1.096034,-0.631478,1.575287,-0.959799,0.285255,-0.090326,1.131756,0.024987,1.254937,-0.098879,1.685594,-0.441015,-0.743204,0.816555,1.453535,1.230915,0.659502,-0.207471,-1.124885,1.054915,0.263851,-0.906435,1.263864,0.048332,0.211484,0.912404,0.145769,1.918973,-0.370218,1.035343,1.410703,-0.653651,-0.496489,0.485162,-0.283391,0.425975,0.744346,1.429698,-1.058599,1.613575,-0.219368,-1.058224,-0.877304,0.404084,1.528637,-0.213243,0.630662,0.093203,0.273019,-1.351241,-1.211005,2.162419,0.592243,-0.009587,0.463209,-0.276363,0.233067,0.892943,-2.696205,0.345526,-3.287736,0.194912,0.721704,1.429703,-0.793404,-1.051918,-0.070013,-1.839509,0.615814,-0.406750,-0.717274,-1.389018,-1.077289,0.053001,0.075631,-0.852768,0.839802,-0.085009,-1.215709,-0.277871,0.190528,0.832015,-2.045363,0.828776,0.169454
2,1.072193,-0.225837,-0.248071,-0.931937,-0.536273,-1.259898,1.412038,0.057953,-2.185032,-0.556375,0.799263,0.641333,-2.193436,0.436186,-0.347765,1.644891,1.809859,-3.869919,-0.301438,-0.447515,1.487495,1.020325,0.732995,0.967562,0.956817,2.207665,-0.414518,1.043236,0.052486,-1.469929,-0.417482,-0.190413,1.103173,0.016437,-0.764251,-0.444657,0.518061,0.302810,-0.126235,1.864022,-0.122029,0.364025,-1.876136,0.031565,-0.533291,0.638918,-1.284157,-0.847103,0.931844,1.444917,-0.636015,0.396275,1.205446,-0.986122,-0.851324,-0.780627,-0.215774,1.944059,0.342010,1.370070,1.285238,0.142632,1.588174,1.304909,1.067904,2.330774,1.751628,0.492946,-0.600396,0.920874,-0.662236,0.187152,-0.848930,0.490690,-0.393960,-1.094003,1.031168,-0.879787,-0.335949,1.342463,-0.162084,-2.053880,0.238026,0.076526,0.733880,-2.527900,0.795950,-1.367659,0.161949,-0.404411,0.217757,0.804863,-0.691985,0.777926,-0.213608,-1.569147,-1.542918,-1.620071,-1.127629,0.320905
3,-0.456109,0.721613,1.323648,1.946415,-0.574469,1.101483,-0.285606,0.688400,-0.634971,-1.535189,-0.314742,2.112819,-0.306209,0.195311,-2.312502,-0.711654,-0.210235,-1.631480,0.394211,-1.039672,0.181436,1.576412,0.523647,1.001608,0.095966,0.295656,0.955692,0.918324,1.056865,1.312108,0.021079,-0.583975,-0.648752,0.379128,0.930245,1.337610,-0.003536,-0.216840,-1.461144,-2.172934,-0.262747,-0.584545,1.088886,-1.596303,-2.023146,1.816994,0.616903,0.822793,1.228845,1.424485,0.085608,0.681311,1.067088,1.541028,-0.013434,-1.109092,-1.022712,1.098081,-0.337786,-1.693251,1.256905,0.977131,1.991856,0.224048,0.114668,0.131897,-0.034806,3.196861,0.555257,0.126024,-0.214003,-0.132092,0.580908,0.906497,1.782913,0.420300,-2.123147,0.294075,1.132817,-0.385427,0.765166,-0.170948,-0.480066,-0.284207,-1.867765,0.203592,-1.566268,-0.887184,-0.680697,-1.112205,0.6185

## HTML Escaping

In [56]:
df4 = pd.DataFrame([["<div></div>", "'&other'", "<span></span>"]])
df4.style

,0,1,2
0,,'&other',


In [57]:
df4.style.format(escape="html")

,0,1,2
0,<div></div>,'&other',<span></span>


In [58]:
df4.style.format("<a href='https://pandas.pydata.org' target='_blank'>{}</a>", escape="html")

,0,1,2
0,<div></div>,'&other',<span></span>


## Export to Excel

In [ ]:
df2.style.map(
    style_negative,
    props="color: red;",
).highlight_max(axis=0).to_excel(
    "styled.xlsx",
    engine="openpyxl",
)